## SQL Transformation

Building silver- and gold-type datasets from the raw (bronze-) dataset

Out goal is prediction of energy usage, so we need to identify/derive/engineer
features that are for sure available at the time of prediction.

#Mental model

RAW / BRONZE
        ↓
preserve source

SILVER
        ↓
rename
type cast
standardize
normalize NULLs
flag invalid records
resolve duplicates
derive universal clean fields
        ↓
reusable trusted data

GOLD
        ↓
aggregate
join
lag
rolling window
business rules
domain features
        ↓
specific analytical use case

And 5 questions after each tranformation

1. What does one row represent now?

2. Did my row count change?
   Should it have?

3. Is my key still unique?

4. Did I create unexpected NULLs?

5. Did I accidentally use information from the future?

In [26]:
#Starting from the raw table
import duckdb

con = duckdb.connect("Steel_energy.duckdb")

In [27]:
#Renaming columns
# We are going to create a silver-table
# We first convert the datetime format and remname the columns

con.sql("""
CREATE OR REPLACE TABLE silver_energy_usage AS

SELECT
    TRY_STRPTIME(
        date,
        '%d/%m/%Y %H:%M'
    ) AS timestamp,

    Usage_kWh AS usage_kwh,

    "Lagging_Current_Reactive.Power_kVarh"
        AS lagging_reactive_power_kvarh,

    Leading_Current_Reactive_Power_kVarh
        AS leading_reactive_power_kvarh,

    "CO2(tCO2)"
        AS co2_tco2,

    Lagging_Current_Power_Factor
        AS lagging_power_factor,

    Leading_Current_Power_Factor
        AS leading_power_factor,

    NSM
        AS seconds_from_midnight,

    WeekStatus
        AS week_status,

    Day_of_week
        AS day_of_week,

    Load_Type
        AS load_type

FROM raw_energy_usage
""")

In [28]:
#Normalizing text categories
con.sql("""
    SELECT 
        load_type,
        COUNT(*) AS observations
    FROM raw_energy_usage
    GROUP BY load_type
""")

┌──────────────┬──────────────┐
│  Load_Type   │ observations │
│   varchar    │    int64     │
├──────────────┼──────────────┤
│ Medium_Load  │         9696 │
│ Maximum_Load │         7272 │
│ Light_Load   │        18072 │
└──────────────┴──────────────┘

#It looks like the categories are well written. But they might have errors such as typos or different capital letters. We can normalize them as follows, for each categorical column.

In [29]:
con.sql("""
    SELECT
        CASE
            WHEN LOWER(TRIM(Load_Type)) = 'light_load'
                THEN 'Light_Load'
        
            WHEN LOWER(TRIM(Load_Type)) = 'medium_load'
                THEN 'Medium_Load'
        
            WHEN LOWER(TRIM(Load_Type)) = 'maximum_load'
                THEN 'Maximum_Load'
        
            ELSE TRIM(Load_Type)
        END AS load_type
    FROM silver_energy_usage
""")

┌────────────────────────┐
│       load_type        │
│        varchar         │
├────────────────────────┤
│ Light_Load             │
│ Light_Load             │
│ Light_Load             │
│ Light_Load             │
│ Light_Load             │
│ Light_Load             │
│ Light_Load             │
│ Light_Load             │
│ Light_Load             │
│ Light_Load             │
│     ·                  │
│     ·                  │
│     ·                  │
│ Light_Load             │
│ Light_Load             │
│ Light_Load             │
│ Light_Load             │
│ Light_Load             │
│ Light_Load             │
│ Light_Load             │
│ Light_Load             │
│ Light_Load             │
│ Light_Load             │
└────────────────────────┘
          ? rows        
  (>9999 rows, 20 shown) 

In [30]:
#Normalizing fake NULL representations
#if a source contains '', N/A, NA, unknown, etc, instead of NULL

con.sql("""
    SELECT
        CASE
            WHEN LOWER(TRIM(Load_Type))
                 IN ('', 'na', 'n/a', 'unknown')
            THEN NULL
        
            ELSE TRIM(Load_Type)
        END AS load_type
    FROM silver_energy_usage
""")

┌────────────────────────┐
│       load_type        │
│        varchar         │
├────────────────────────┤
│ Light_Load             │
│ Light_Load             │
│ Light_Load             │
│ Light_Load             │
│ Light_Load             │
│ Light_Load             │
│ Light_Load             │
│ Light_Load             │
│ Light_Load             │
│ Light_Load             │
│     ·                  │
│     ·                  │
│     ·                  │
│ Light_Load             │
│ Light_Load             │
│ Light_Load             │
│ Light_Load             │
│ Light_Load             │
│ Light_Load             │
│ Light_Load             │
│ Light_Load             │
│ Light_Load             │
│ Light_Load             │
└────────────────────────┘
          ? rows        
  (>9999 rows, 20 shown) 

In [31]:
#Derive calendar variables
#We can derive year, month, day, hour, weekday, weekend
#These variables are useful for time-based predictions
#For example energy consumption prediction at a given time of the day/week

con.sql("""
SELECT
    timestamp,

    EXTRACT(YEAR FROM timestamp) AS year,
    EXTRACT(MONTH FROM timestamp) AS month,
    EXTRACT(DAY FROM timestamp) AS day,
    EXTRACT(HOUR FROM timestamp) AS hour,

    STRFTIME(timestamp, '%A') AS calculated_weekday

FROM silver_energy_usage

LIMIT 10
""")

┌─────────────────────┬───────┬───────┬───────┬───────┬────────────────────┐
│      timestamp      │ year  │ month │  day  │ hour  │ calculated_weekday │
│      timestamp      │ int64 │ int64 │ int64 │ int64 │      varchar       │
├─────────────────────┼───────┼───────┼───────┼───────┼────────────────────┤
│ 2018-01-01 00:15:00 │  2018 │     1 │     1 │     0 │ Monday             │
│ 2018-01-01 00:30:00 │  2018 │     1 │     1 │     0 │ Monday             │
│ 2018-01-01 00:45:00 │  2018 │     1 │     1 │     0 │ Monday             │
│ 2018-01-01 01:00:00 │  2018 │     1 │     1 │     1 │ Monday             │
│ 2018-01-01 01:15:00 │  2018 │     1 │     1 │     1 │ Monday             │
│ 2018-01-01 01:30:00 │  2018 │     1 │     1 │     1 │ Monday             │
│ 2018-01-01 01:45:00 │  2018 │     1 │     1 │     1 │ Monday             │
│ 2018-01-01 02:00:00 │  2018 │     1 │     1 │     2 │ Monday             │
│ 2018-01-01 02:15:00 │  2018 │     1 │     1 │     2 │ Monday             │

In [32]:
#Derived variables from timestamps
#For example work shifts

con.sql("""
SELECT
    timestamp,

    CASE
        WHEN EXTRACT(HOUR FROM timestamp) < 8
            THEN 'Shift_1'

        WHEN EXTRACT(HOUR FROM timestamp) < 16
            THEN 'Shift_2'

        ELSE 'Shift_3'
    END AS shift

FROM silver_energy_usage

LIMIT 20
""")

┌─────────────────────┬─────────┐
│      timestamp      │  shift  │
│      timestamp      │ varchar │
├─────────────────────┼─────────┤
│ 2018-01-01 00:15:00 │ Shift_1 │
│ 2018-01-01 00:30:00 │ Shift_1 │
│ 2018-01-01 00:45:00 │ Shift_1 │
│ 2018-01-01 01:00:00 │ Shift_1 │
│ 2018-01-01 01:15:00 │ Shift_1 │
│ 2018-01-01 01:30:00 │ Shift_1 │
│ 2018-01-01 01:45:00 │ Shift_1 │
│ 2018-01-01 02:00:00 │ Shift_1 │
│ 2018-01-01 02:15:00 │ Shift_1 │
│ 2018-01-01 02:30:00 │ Shift_1 │
│ 2018-01-01 02:45:00 │ Shift_1 │
│ 2018-01-01 03:00:00 │ Shift_1 │
│ 2018-01-01 03:15:00 │ Shift_1 │
│ 2018-01-01 03:30:00 │ Shift_1 │
│ 2018-01-01 03:45:00 │ Shift_1 │
│ 2018-01-01 04:00:00 │ Shift_1 │
│ 2018-01-01 04:15:00 │ Shift_1 │
│ 2018-01-01 04:30:00 │ Shift_1 │
│ 2018-01-01 04:45:00 │ Shift_1 │
│ 2018-01-01 05:00:00 │ Shift_1 │
└─────────────────────┴─────────┘
  20 rows             2 columns

In [33]:
#Create aggregations
#For example average, min, max values for a given category

con.sql("""
SELECT
    load_type,

    COUNT(*) AS observations,

    AVG(usage_kwh) AS avg_usage_kwh,

    MIN(usage_kwh) AS min_usage_kwh,

    MAX(usage_kwh) AS max_usage_kwh

FROM silver_energy_usage

GROUP BY load_type

ORDER BY avg_usage_kwh DESC
""")

┌──────────────┬──────────────┬────────────────────┬───────────────┬───────────────┐
│  load_type   │ observations │   avg_usage_kwh    │ min_usage_kwh │ max_usage_kwh │
│   varchar    │    int64     │       double       │    double     │    double     │
├──────────────┼──────────────┼────────────────────┼───────────────┼───────────────┤
│ Maximum_Load │         7272 │  59.26531353135312 │          2.92 │        151.67 │
│ Medium_Load  │         9696 │ 38.445393976897975 │          2.52 │        157.18 │
│ Light_Load   │        18072 │  8.626206839310065 │           0.0 │        140.29 │
└──────────────┴──────────────┴────────────────────┴───────────────┴───────────────┘

In [34]:
#Daily aggregation
con.sql("""
SELECT
    CAST(timestamp AS DATE) AS date,

    SUM(usage_kwh) AS total_usage_kwh,

    AVG(usage_kwh) AS avg_interval_usage_kwh,

    MAX(usage_kwh) AS max_interval_usage_kwh,

    COUNT(*) AS observations

FROM silver_energy_usage

GROUP BY date

ORDER BY date
""")

┌────────────┬────────────────────┬────────────────────────┬────────────────────────┬──────────────┐
│    date    │  total_usage_kwh   │ avg_interval_usage_kwh │ max_interval_usage_kwh │ observations │
│    date    │       double       │         double         │         double         │    int64     │
├────────────┼────────────────────┼────────────────────────┼────────────────────────┼──────────────┤
│ 2018-01-01 │  351.8599999999999 │     3.6652083333333323 │                   4.28 │           96 │
│ 2018-01-02 │ 3950.4300000000003 │     41.150312500000005 │                 147.46 │           96 │
│ 2018-01-03 │ 3561.0500000000006 │      37.09427083333334 │                 140.51 │           96 │
│ 2018-01-04 │            4977.72 │               51.85125 │                 144.29 │           96 │
│ 2018-01-05 │  4683.400000000001 │      48.78541666666667 │                 146.34 │           96 │
│ 2018-01-06 │             372.96 │                  3.885 │                   4.54 │      

#This calculation is actually correct only if Usage_kWh represents the energy consumed during each measurement interval

In [35]:
#Let's build a Gold table
#The gold table depends on your goal
#If your goal is about daily energy analytics, we need a table with daily data
#One row corresponds to one day
con.sql("""
CREATE OR REPLACE TABLE gold_daily_energy AS

SELECT
    CAST(timestamp AS DATE) AS date,

    SUM(usage_kwh) AS total_usage_kwh,

    AVG(usage_kwh) AS avg_usage_kwh,

    MAX(usage_kwh) AS peak_usage_kwh,

    AVG(lagging_power_factor)
        AS avg_lagging_power_factor,

    AVG(co2_tco2)
        AS avg_co2_tco2,

    COUNT(*) AS observations

FROM silver_energy_usage

GROUP BY date
""")

#Window functions
#For forecasting, previous measurements are often predictors.
#LAG() function is great. It can give you the previous observation related to a given one
#LAG(column,n) gives you the value n rows before the current observation
#By placing that previous observation in the actual row, you can use it as a feature with information about the past

In [36]:
con.sql("""
SELECT
    timestamp,
    usage_kwh,

    LAG(usage_kwh, 1) OVER (
        ORDER BY timestamp
    ) AS usage_15min_ago,

    LAG(usage_kwh, 2) OVER (
        ORDER BY timestamp
    ) AS usage_30min_ago,

    LAG(usage_kwh, 4) OVER (
        ORDER BY timestamp
    ) AS usage_1h_ago

FROM silver_energy_usage

ORDER BY timestamp

LIMIT 20
""")

┌─────────────────────┬───────────┬─────────────────┬─────────────────┬──────────────┐
│      timestamp      │ usage_kwh │ usage_15min_ago │ usage_30min_ago │ usage_1h_ago │
│      timestamp      │  double   │     double      │     double      │    double    │
├─────────────────────┼───────────┼─────────────────┼─────────────────┼──────────────┤
│ 2018-01-01 00:00:00 │      3.42 │            NULL │            NULL │         NULL │
│ 2018-01-01 00:15:00 │      3.17 │            3.42 │            NULL │         NULL │
│ 2018-01-01 00:30:00 │       4.0 │            3.17 │            3.42 │         NULL │
│ 2018-01-01 00:45:00 │      3.24 │             4.0 │            3.17 │         NULL │
│ 2018-01-01 01:00:00 │      3.31 │            3.24 │             4.0 │         3.42 │
│ 2018-01-01 01:15:00 │      3.82 │            3.31 │            3.24 │         3.17 │
│ 2018-01-01 01:30:00 │      3.28 │            3.82 │            3.31 │          4.0 │
│ 2018-01-01 01:45:00 │       3.6 │        

In [37]:
#Rolling average
#Average values about previous observations
#e.g. average energy consumption during the previous hour 
#In our case we have 15 min records, so we need an average of the previous 4 obs. 
#We need to exclude the current observation to avoid leakage
con.sql("""
SELECT
    timestamp,
    usage_kwh,

    AVG(usage_kwh) OVER (
        ORDER BY timestamp
        ROWS BETWEEN 4 PRECEDING
                 AND 1 PRECEDING
    ) AS avg_usage_previous_hour

FROM silver_energy_usage

ORDER BY timestamp

LIMIT 20
""")

┌─────────────────────┬───────────┬─────────────────────────┐
│      timestamp      │ usage_kwh │ avg_usage_previous_hour │
│      timestamp      │  double   │         double          │
├─────────────────────┼───────────┼─────────────────────────┤
│ 2018-01-01 00:00:00 │      3.42 │                    NULL │
│ 2018-01-01 00:15:00 │      3.17 │                    3.42 │
│ 2018-01-01 00:30:00 │       4.0 │                   3.295 │
│ 2018-01-01 00:45:00 │      3.24 │                    3.53 │
│ 2018-01-01 01:00:00 │      3.31 │                  3.4575 │
│ 2018-01-01 01:15:00 │      3.82 │                    3.43 │
│ 2018-01-01 01:30:00 │      3.28 │      3.5925000000000002 │
│ 2018-01-01 01:45:00 │       3.6 │                  3.4125 │
│ 2018-01-01 02:00:00 │       3.6 │                  3.5025 │
│ 2018-01-01 02:15:00 │      3.28 │      3.5749999999999997 │
│ 2018-01-01 02:30:00 │      3.78 │                    3.44 │
│ 2018-01-01 02:45:00 │      3.46 │                   3.565 │
│ 2018-0

In [38]:
#Rolling maximum
#Similar idea
con.sql("""
SELECT
    timestamp,
    usage_kwh,

    MAX(usage_kwh) OVER (
        ORDER BY timestamp
        ROWS BETWEEN 4 PRECEDING
                 AND 1 PRECEDING
    ) AS max_usage_previous_hour

FROM silver_energy_usage

ORDER BY timestamp

LIMIT 20
""")


┌─────────────────────┬───────────┬─────────────────────────┐
│      timestamp      │ usage_kwh │ max_usage_previous_hour │
│      timestamp      │  double   │         double          │
├─────────────────────┼───────────┼─────────────────────────┤
│ 2018-01-01 00:00:00 │      3.42 │                    NULL │
│ 2018-01-01 00:15:00 │      3.17 │                    3.42 │
│ 2018-01-01 00:30:00 │       4.0 │                    3.42 │
│ 2018-01-01 00:45:00 │      3.24 │                     4.0 │
│ 2018-01-01 01:00:00 │      3.31 │                     4.0 │
│ 2018-01-01 01:15:00 │      3.82 │                     4.0 │
│ 2018-01-01 01:30:00 │      3.28 │                     4.0 │
│ 2018-01-01 01:45:00 │       3.6 │                    3.82 │
│ 2018-01-01 02:00:00 │       3.6 │                    3.82 │
│ 2018-01-01 02:15:00 │      3.28 │                    3.82 │
│ 2018-01-01 02:30:00 │      3.78 │                     3.6 │
│ 2018-01-01 02:45:00 │      3.46 │                    3.78 │
│ 2018-0

In [39]:
#Window function with PARTITION BY
#Imagine we have three categories (e.g. 3 machines)
#Using LAG without selecting the previous obs. from a given machine would be wrong
#Partition by let's you select previous obs. with LAG with a given grouping

con.sql("""
SELECT
    usage_kwh,
    load_type,

    LAG(usage_kwh) OVER (
        PARTITION BY load_type
        ORDER BY timestamp
    )
    
FROM silver_energy_usage

ORDER BY timestamp

LIMIT 20
""")

┌───────────┬────────────┬───────────────────────────────────────────────────────────────────┐
│ usage_kwh │ load_type  │ lag(usage_kwh) OVER (PARTITION BY load_type ORDER BY "timestamp") │
│  double   │  varchar   │                              double                               │
├───────────┼────────────┼───────────────────────────────────────────────────────────────────┤
│      3.42 │ Light_Load │                                                              NULL │
│      3.17 │ Light_Load │                                                              3.42 │
│       4.0 │ Light_Load │                                                              3.17 │
│      3.24 │ Light_Load │                                                               4.0 │
│      3.31 │ Light_Load │                                                              3.24 │
│      3.82 │ Light_Load │                                                              3.31 │
│      3.28 │ Light_Load │                        

In [40]:
#Joining auxiliary tables
#If you have tables with a matching key you can join them
#Or sometimes you create a new auxiliary table
#e.g.
con.sql("""
CREATE OR REPLACE TABLE load_type_info AS

SELECT *
FROM (
    VALUES
        ('Light_Load', 1),
        ('Medium_Load', 2),
        ('Maximum_Load', 3)
)
AS t(load_type, operating_priority)
""")

In [41]:
#Now we can join them
display(
    con.sql("""
SELECT
    s.*,
    l.operating_priority

FROM silver_energy_usage AS s

LEFT JOIN load_type_info AS l

    ON s.load_type = l.load_type
""")
)

┌─────────────────────┬───────────┬──────────────────────────────┬──────────────────────────────┬──────────┬──────────────────────┬──────────────────────┬───────────────────────┬─────────────┬─────────────┬────────────┬────────────────────┐
│      timestamp      │ usage_kwh │ lagging_reactive_power_kvarh │ leading_reactive_power_kvarh │ co2_tco2 │ lagging_power_factor │ leading_power_factor │ seconds_from_midnight │ week_status │ day_of_week │ load_type  │ operating_priority │
│      timestamp      │  double   │            double            │            double            │  double  │        double        │        double        │         int64         │   varchar   │   varchar   │  varchar   │       int32        │
├─────────────────────┼───────────┼──────────────────────────────┼──────────────────────────────┼──────────┼──────────────────────┼──────────────────────┼───────────────────────┼─────────────┼─────────────┼────────────┼────────────────────┤
│ 2018-01-01 00:15:00 │      3.17 │ 

#left join keeps the same number of observations of the left table, the one you define with FROM. The right table is the one you type in LEFT JOIN. l in this case is the joined table

In [42]:
#Always validate after a join
#With left join you need the same nr of rows of left tab. before and after joining 
con.sql("""
SELECT COUNT(*)

FROM silver_energy_usage AS s

LEFT JOIN load_type_info AS l
    ON s.load_type = l.load_type
""")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│        35040 │
└──────────────┘

In [43]:
#Validate unmatched joins
#If keys are not matching, the LEFT join returns a NULL
#Validate the NULL amounts

con.sql("""
SELECT
    s.load_type,
    COUNT(*) AS observations

FROM silver_energy_usage AS s

LEFT JOIN load_type_info AS l
    ON s.load_type = l.load_type

WHERE l.load_type IS NULL

GROUP BY s.load_type
""")

┌───────────┬──────────────┐
│ load_type │ observations │
│  varchar  │    int64     │
└───────────┴──────────────┘
           0 rows         

In [45]:
#Gold table for forecasting
con.sql("""
CREATE OR REPLACE TABLE gold_energy_features AS

WITH historical_features AS (

    SELECT
        timestamp,

        usage_kwh,

        EXTRACT(HOUR FROM timestamp)
            AS hour,

        EXTRACT(MONTH FROM timestamp)
            AS month,

        STRFTIME(timestamp, '%A')
            AS day_of_week,

        week_status,
        load_type,

        LAG(usage_kwh, 1) OVER (
            ORDER BY timestamp
        ) AS usage_15min_ago,

        LAG(usage_kwh, 4) OVER (
            ORDER BY timestamp
        ) AS usage_1h_ago,

        LAG(usage_kwh, 96) OVER (
            ORDER BY timestamp
        ) AS usage_lag_96,

        AVG(usage_kwh) OVER (
            ORDER BY timestamp
            ROWS BETWEEN 4 PRECEDING
                     AND 1 PRECEDING
        ) AS avg_usage_previous_hour

    FROM silver_energy_usage
)

SELECT *
FROM historical_features
""")

In [22]:
#CO2, reactive power, power factors were excluded.
#not sure if they are avaialble before prediction
#they could be used for an estimation problem (soft sensor)
#not for future forecasting
#to use these variables, you could define past rolling averages or use LAG

In [23]:
#Validate the gold table
con.sql("""
SELECT
    COUNT(*) - COUNT(usage_15min_ago)
        AS null_lag_15m,

    COUNT(*) - COUNT(usage_1h_ago)
        AS null_lag_1h,

    COUNT(*) - COUNT(avg_usage_previous_hour)
        AS null_rolling_1h

FROM gold_energy_features
""")

┌──────────────┬─────────────┬─────────────────┐
│ null_lag_15m │ null_lag_1h │ null_rolling_1h │
│    int64     │    int64    │      int64      │
├──────────────┼─────────────┼─────────────────┤
│            1 │           4 │               1 │
└──────────────┴─────────────┴─────────────────┘

#Some NULLs at the beginning expected due to LAG/AVG using initial values non existing

#Estimation = infer a value for the present from other information available at the same time.
#Forecasting = predict a value that has not happened yet.

#Two completely different ML problems, that require two different golden tables

In [47]:
con.close()